# JobGuard AI – AI-Powered Fake Job & Internship Detection Platform
## Complete Machine Learning Pipeline, Benchmark Comparison & Explainable AI
**Academic Project Level:** B.Tech Computer Science / Artificial Intelligence Capstone Project  
**Target Domain:** Natural Language Processing (NLP), Text Classification, Supervised Learning, Fraud Analytics  
**Primary Dataset:** EMSCAD (Employment Scam Aegean Dataset) — 17,880 historical job listings  
**Target Column:** `fraudulent` (0 = Legitimate Opportunity, 1 = Potentially Fraudulent / Scam)  
**Backend Compatibility:** Fully compatible with JobGuard AI FastAPI REST API (`jobguard_pipeline_v1.joblib`)

---

### Project Structure & Methodology
1. **Environment Setup & Reproducibility:** Library versions, random seed pinning (`RANDOM_STATE = 42`).
2. **Dataset Ingestion & Auto-Detection:** Colab file upload with local path fallback; schema integrity verification.
3. **Dataset Schema, Data Types & Duplicate Check:** Dimensions, missing columns, duplicate record detection, target distribution (17,014 Legitimate vs 866 Fraudulent).
4. **Comprehensive Exploratory Data Analysis (EDA):**
   - Class imbalance visualization (Bar & Donut charts).
   - Missing value frequencies across attributes.
   - Company logo, experience level, remote status, and employment type breakdowns.
   - Geographic distribution (top country codes) and high-risk industry analysis.
   - Job description character length KDE density plot.
   - Structured EDA summary JSON export (`eda_summary.json`).
5. **Text Cleaning & NLP Normalization:** HTML stripping, URL/email token replacement, lowercase conversion, and special character filtering.
6. **Scikit-Learn Transformer Architecture:** Custom `TextCombiner` and `StructuredFeatureExtractor` adhering to estimator contracts (`is_fitted_ = True`, `__sklearn_is_fitted__`).
7. **Data Leakage Prevention:** Stratified 80/20 train-test partition prior to feature extraction.
8. **Production Model Pipeline Construction:** Composite `FeatureUnion` combining 10,000 sublinear TF-IDF word & character n-grams with 7 structured metadata features, feeding into class-balanced Logistic Regression.
9. **Stratified 5-Fold Cross-Validation:** F1-score evaluation across training folds.
10. **Primary Model Training & Evaluation:** Test set evaluation reporting Accuracy, Fraud Precision, Fraud Recall, Fraud F1-Score, ROC-AUC, and PR-AUC.
11. **Multi-Model Benchmark Comparison:**
   - Model 1: Balanced Logistic Regression (Production Pipeline)
   - Model 2: Calibrated Linear SVM
   - Model 3: Multinomial Naive Bayes
   - Model 4: Random Forest Classifier (Structured Features)
   - Tabular comparison and visual exhibit (`model_comparison.csv`).
12. **Diagnostic Visualizations:** Annotated Confusion Matrix heatmap, Multi-Model ROC Curves, and Precision-Recall Curves (`confusion_matrix.png`).
13. **Explainable AI (XAI) Feature Attribution:** Extraction and visualization of top positive (fraud-pushing) and negative (legitimacy-pushing) TF-IDF feature weights.
14. **Internship Scam Rule Engine & Dual-Engine Scoring:** Heuristic pattern checks for student scams (fees, guaranteed offers, Telegram/WhatsApp contacts, missing company profile) with composite scoring: 60% ML + 40% Rules.
15. **Real-Time Inference Demos:** 3 realistic test cases demonstrating risk assessment.
16. **Pipeline Serialization & Colab Download:** Packaging into `jobguard_ai_artifacts.zip` and one-click Google Colab download.
17. **Academic Presentation & Deployment Guide:** Integrating the trained pipeline into the local full-stack JobGuard AI platform.


## 1. Environment Setup & Dependency Installation
Install required libraries, display exact package versions, and set a fixed random seed for 100% reproducible results.


In [ ]:
# Install dependencies in Google Colab environment
!pip install -q scikit-learn pandas numpy matplotlib seaborn nltk joblib scipy

import sys
import os
import re
import html
import json
import time
import zipfile
import shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# Set reproducible random state seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("=" * 60)
print("JOBGUARD AI ENVIRONMENT INITIALIZED")
print("=" * 60)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"Pandas Version:       {pd.__version__}")
print(f"NumPy Version:        {np.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Matplotlib Version:   {matplotlib.__version__}")
print(f"Joblib Version:       {joblib.__version__}")
print(f"Random State Seed:    {RANDOM_STATE}")
print("=" * 60)


## 2. Dataset Ingestion & Auto-Detection
Upload `fake_job_postings.csv` (EMSCAD dataset) using the interactive Google Colab file uploader or detect it automatically from local directory paths. If the dataset is not found, execution halts with a helpful instruction (no synthetic or fabricated data).


In [ ]:
# Dataset Upload / Detection Cell
print("Checking for EMSCAD dataset file: fake_job_postings.csv ...")

possible_dataset_paths = [
    'fake_job_postings.csv',
    'dataset/fake_job_postings.csv',
    '../dataset/fake_job_postings.csv',
    '../../dataset/fake_job_postings.csv'
]

dataset_path = None
for p in possible_dataset_paths:
    if os.path.exists(p) and os.path.isfile(p):
        dataset_path = p
        break

# If not found locally, trigger Google Colab upload prompt
if dataset_path is None:
    try:
        from google.colab import files
        print("\n[ACTION REQUIRED] Please upload 'fake_job_postings.csv' from your local computer:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.csv'):
                dataset_path = fn
                break
    except ImportError:
        pass

if dataset_path is None or not os.path.exists(dataset_path):
    raise FileNotFoundError(
        "CRITICAL ERROR: 'fake_job_postings.csv' not found!\n"
        "Please download the EMSCAD dataset (Real or Fake Job Postings) and upload it here before continuing."
    )

print(f"\nLoading dataset from verified path: {dataset_path}")
df = pd.read_csv(dataset_path)
print("Dataset successfully loaded into memory!")
print(f"Dimensions: {df.shape[0]:,} rows and {df.shape[1]} columns.")


## 3. Dataset Schema, Data Types & Target Integrity Validation
Validate all 14 required attributes, inspect dimensions, data types, duplicate records, and verify the target class distribution (`fraudulent`: 0 = Legitimate, 1 = Fraudulent).


In [ ]:
# Schema Validation, Data Types, and Class Distribution Inspection
REQUIRED_COLUMNS = [
    "title", "company_profile", "description", "requirements", "benefits",
    "telecommuting", "has_company_logo", "has_questions", "employment_type",
    "required_experience", "required_education", "industry", "function", "fraudulent"
]

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"CRITICAL ERROR: Dataset is missing required columns: {missing_cols}")

print("Schema Validation: PASSED (All 14 required attributes verified).")

# Dataset dimensions and duplicate check
n_rows, n_cols = df.shape
n_duplicates = df.duplicated().sum()
print(f"Dimensions: {n_rows:,} rows, {n_cols} columns")
print(f"Duplicate records: {n_duplicates:,}")

# Clean and validate fraudulent binary target column
df['fraudulent'] = pd.to_numeric(df['fraudulent'], errors='coerce')
df = df.dropna(subset=['fraudulent'])
df['fraudulent'] = df['fraudulent'].astype(int)

counts = df['fraudulent'].value_counts().to_dict()
n_legit = counts.get(0, 0)
n_fraud = counts.get(1, 0)
n_total = len(df)
fraud_rate = (n_fraud / n_total) * 100

print("\n" + "=" * 50)
print("EMSCAD DATASET TARGET BREAKDOWN")
print("=" * 50)
print(f"Total Valid Listings:    {n_total:,}")
print(f"Legitimate Postings (0): {n_legit:,} ({100 - fraud_rate:.2f}%)")
print(f"Fraudulent Postings (1): {n_fraud:,} ({fraud_rate:.2f}%)")
print(f"Class Imbalance Ratio:   1 : {n_legit / max(n_fraud, 1):.1f}")
print("=" * 50)

print("\nColumn Data Types:")
print(df.dtypes)

print("\nFirst 5 Records:")
display(df.head(5))


## 4. Comprehensive Exploratory Data Analysis (EDA)
Inspect class distribution, missing value profiles, corporate logo correlation, experience levels, employment types, geographical locations, industries, and description length distributions.


In [ ]:
# EDA Visual 1: Target Class Imbalance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Bar Chart
sns.barplot(x=['Legitimate (0)', 'Fraudulent (1)'], y=[n_legit, n_fraud], palette=['#10B981', '#EF4444'], ax=ax1)
ax1.set_title('Target Distribution: Legitimate vs Fraudulent Job Postings', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Postings')
for i, v in enumerate([n_legit, n_fraud]):
    ax1.text(i, v + 250, f"{v:,} ({v/n_total*100:.2f}%)", ha='center', fontweight='bold')

# Subplot 2: Donut Chart
ax2.pie([n_legit, n_fraud], labels=['Legitimate (95.16%)', 'Fraudulent (4.84%)'],
        colors=['#10B981', '#EF4444'], autopct='%1.1f%%', startangle=140,
        wedgeprops=dict(width=0.4, edgecolor='w'))
ax2.set_title('Class Imbalance Proportion', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key Finding: Only ~4.8% of postings are fraudulent. A naive model predicting 'Legitimate' for every post")
print("would achieve 95.16% accuracy while identifying ZERO scams. Optimizing Fraud Recall is imperative.")


In [ ]:
# EDA Visual 2: Missing Values Analysis Across Attributes
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=True)

plt.figure(figsize=(10, 6))
missing.plot(kind='barh', color='#F59E0B')
plt.title('Missing Value Count per Feature in EMSCAD Dataset', fontsize=12, fontweight='bold')
plt.xlabel('Number of Missing Values')
for index, value in enumerate(missing):
    plt.text(value + 100, index, f"{value:,} ({value/len(df)*100:.1f}%)", va='center', fontsize=9)
plt.tight_layout()
plt.show()

print("Observation: High missing rates in salary_range, department, and company_profile.")


In [ ]:
# EDA Visual 3: Company Logo, Experience, Remote & Employment Type Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Company Logo vs Fraud
logo_crosstab = pd.crosstab(df['has_company_logo'], df['fraudulent'], normalize='index') * 100
logo_crosstab.plot(kind='bar', stacked=True, color=['#10B981', '#EF4444'], ax=axes[0, 0])
axes[0, 0].set_title('Fraud Rate by Company Logo Presence', fontweight='bold')
axes[0, 0].set_xticklabels(['Logo Missing (0)', 'Logo Present (1)'], rotation=0)
axes[0, 0].set_ylabel('Percentage (%)')
axes[0, 0].legend(['Legit (0)', 'Fraud (1)'])

# 2. Required Experience vs Fraud
exp_fraud = df.groupby('required_experience')['fraudulent'].mean().sort_values(ascending=False) * 100
sns.barplot(x=exp_fraud.values, y=exp_fraud.index, palette='Reds_r', ax=axes[0, 1])
axes[0, 1].set_title('Fraud Probability by Experience Level', fontweight='bold')
axes[0, 1].set_xlabel('Fraud Rate (%)')

# 3. Telecommuting (Remote) vs Fraud
tele_crosstab = pd.crosstab(df['telecommuting'], df['fraudulent'], normalize='index') * 100
tele_crosstab.plot(kind='bar', stacked=True, color=['#10B981', '#EF4444'], ax=axes[1, 0])
axes[1, 0].set_title('Fraud Rate by Remote / Telecommuting Status', fontweight='bold')
axes[1, 0].set_xticklabels(['On-site (0)', 'Remote (1)'], rotation=0)
axes[1, 0].set_ylabel('Percentage (%)')
axes[1, 0].legend(['Legit (0)', 'Fraud (1)'])

# 4. Employment Type vs Fraud
emp_fraud = df.groupby('employment_type')['fraudulent'].mean().sort_values(ascending=False) * 100
sns.barplot(x=emp_fraud.values, y=emp_fraud.index, palette='Oranges_r', ax=axes[1, 1])
axes[1, 1].set_title('Fraud Probability by Employment Type', fontweight='bold')
axes[1, 1].set_xlabel('Fraud Rate (%)')

plt.tight_layout()
plt.show()


In [ ]:
# EDA Visual 4: Geographic Location & Industry Fraud Breakdown
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Top Countries by Job Volume & Fraud Rate
df['country'] = df['location'].fillna('').astype(str).apply(lambda x: x.split(',')[0].strip() if ',' in x else x.strip())
top_countries = df['country'].value_counts().head(10).index
country_fraud = df[df['country'].isin(top_countries)].groupby('country')['fraudulent'].mean().sort_values(ascending=False) * 100
sns.barplot(x=country_fraud.values, y=country_fraud.index, palette='Purples_r', ax=axes[0])
axes[0].set_title('Fraud Probability in Top 10 Job Locations (Country Codes)', fontweight='bold')
axes[0].set_xlabel('Fraud Rate (%)')

# 2. Top Industries by Fraud Rate (minimum 50 postings)
ind_counts = df['industry'].value_counts()
valid_inds = ind_counts[ind_counts >= 50].index
ind_fraud = df[df['industry'].isin(valid_inds)].groupby('industry')['fraudulent'].mean().sort_values(ascending=False).head(10) * 100
sns.barplot(x=ind_fraud.values, y=ind_fraud.index, palette='Blues_r', ax=axes[1])
axes[1].set_title('Top 10 High-Risk Industries (>=50 Postings)', fontweight='bold')
axes[1].set_xlabel('Fraud Rate (%)')

plt.tight_layout()
plt.show()


In [ ]:
# EDA Visual 5: Text Length Distribution (Description Characters)
df['desc_length'] = df['description'].fillna('').astype(str).str.len()

plt.figure(figsize=(10, 5))
sns.kdeplot(df[df['fraudulent'] == 0]['desc_length'], label='Legitimate (0)', color='#10B981', fill=True, alpha=0.3)
sns.kdeplot(df[df['fraudulent'] == 1]['desc_length'], label='Fraudulent (1)', color='#EF4444', fill=True, alpha=0.3)
plt.title('Job Description Character Length Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Character Count')
plt.ylabel('Density')
plt.xlim(0, 4000)
plt.legend()
plt.tight_layout()
plt.show()

legit_median = df[df['fraudulent'] == 0]['desc_length'].median()
fraud_median = df[df['fraudulent'] == 1]['desc_length'].median()
print(f"Median Description Length - Legitimate: {legit_median:.0f} chars | Fraudulent: {fraud_median:.0f} chars")

# Export EDA summary JSON
eda_summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "total_rows": int(len(df)),
    "total_columns": int(len(df.columns)),
    "duplicate_rows": int(n_duplicates),
    "legitimate_count": int(n_legit),
    "fraudulent_count": int(n_fraud),
    "fraudulent_percentage": round(fraud_rate, 2),
    "class_imbalance_ratio": f"1 : {round(n_legit / max(n_fraud, 1), 1)}",
    "missing_values": {col: int(cnt) for col, cnt in df.isnull().sum().items() if cnt > 0}
}

with open("eda_summary.json", "w") as f:
    json.dump(eda_summary, f, indent=2)

print("EDA Summary successfully exported to 'eda_summary.json'.")


## 5. Text Cleaning & Natural Language Preprocessing
Vectorized text normalization: HTML stripping, URL and email replacement with tokens, special character removal, and whitespace normalization.


In [ ]:
# Vectorized Text Normalization Functions
def clean_html(text: str) -> str:
    """Strips HTML tags and converts URLs/emails to generic tokens."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    # Replace URLs with 'url' token before stripping tags
    text = re.sub(r"https?://\S+|www\.\S+", " url ", text)
    # Strip HTML tags
    clean_re = re.compile(r"<[^>]+>")
    text = clean_re.sub(" ", text)
    # Replace emails with 'email' token
    text = re.sub(r"\S+@\S+", " email ", text)
    # Normalize whitespaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_text(text: str) -> str:
    """Normalizes text for TF-IDF tokenization."""
    if not isinstance(text, str):
        return ""
    text = clean_html(text).lower()
    text = re.sub(r"[^a-z0-9\s.,!?-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Demonstration of text cleaner on a sample job snippet
sample_raw = """<p><b>URGENT HIRING!</b> Earn $500/day. Visit <a href='https://scam-job.example.com'>here</a> or email hr@faketech.com. 100% guarantee!!</p>"""
print("Raw Input Snippet:")
print(sample_raw)
print("\nCleaned & Normalized Output:")
print(normalize_text(sample_raw))


## 6. Scikit-Learn Custom Pipeline Transformers
Implement `TextCombiner` and `StructuredFeatureExtractor` compatible with Scikit-Learn pipelines. Both implement `self.is_fitted_ = True` and `__sklearn_is_fitted__()` to adhere to estimator contracts and prevent `NotFittedError`.


In [ ]:
# Custom Scikit-Learn Transformers
class TextCombiner(BaseEstimator, TransformerMixin):
    """Combines and cleans textual fields into a unified series."""
    def __init__(self, text_columns=None):
        if text_columns is None:
            self.text_columns = ["title", "company_profile", "description", "requirements", "benefits"]
        else:
            self.text_columns = text_columns

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def __sklearn_is_fitted__(self):
        return getattr(self, "is_fitted_", False)

    def transform(self, X):
        if isinstance(X, dict):
            X = pd.DataFrame([X])
        elif isinstance(X, list):
            X = pd.DataFrame(X)
        elif not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        combined = []
        for col in self.text_columns:
            if col in X.columns:
                s = X[col].fillna("").astype(str)
            else:
                s = pd.Series("", index=X.index)
            combined.append(s)

        result = combined[0]
        for c in combined[1:]:
            result = result + " " + c

        return [normalize_text(t) or "empty" for t in result.values]


class StructuredFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extracts structured categorical and length-based metadata features."""
    def fit(self, X, y=None):
        self.is_fitted_ = True
        self.n_features_in_ = 7
        return self

    def __sklearn_is_fitted__(self):
        return getattr(self, "is_fitted_", False)

    def transform(self, X):
        if isinstance(X, dict):
            X = pd.DataFrame([X])
        elif isinstance(X, list):
            X = pd.DataFrame(X)
        elif not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        has_logo = X.get("has_company_logo", pd.Series(0, index=X.index)).isin([1, "1", True]).astype(np.float32).values
        has_questions = X.get("has_questions", pd.Series(0, index=X.index)).isin([1, "1", True]).astype(np.float32).values
        telecommuting = X.get("telecommuting", pd.Series(0, index=X.index)).isin([1, "1", True]).astype(np.float32).values

        cp = X.get("company_profile", pd.Series("", index=X.index)).fillna("").astype(str).str.strip().str.len()
        has_cp = (cp > 20).astype(np.float32).values

        req = X.get("requirements", pd.Series("", index=X.index)).fillna("").astype(str).str.strip().str.len()
        has_req = (req > 10).astype(np.float32).values

        ben = X.get("benefits", pd.Series("", index=X.index)).fillna("").astype(str).str.strip().str.len()
        has_ben = (ben > 10).astype(np.float32).values

        desc_len = X.get("description", pd.Series("", index=X.index)).fillna("").astype(str).str.len()
        desc_len_norm = np.clip(desc_len.values / 2000.0, 0.0, 5.0).astype(np.float32)

        return np.column_stack([
            has_logo,
            has_questions,
            telecommuting,
            has_cp,
            has_req,
            has_ben,
            desc_len_norm
        ])

print("TextCombiner and StructuredFeatureExtractor successfully defined.")


## 7. Stratified Train-Test Split (Data Leakage Prevention)
Partition dataset into 80% training and 20% testing using stratification. Transformers and vectorizers are fitted strictly on `X_train` and evaluated on `X_test`.


In [ ]:
# Stratified Train-Test Split
X = df[[c for c in df.columns if c not in ['fraudulent', 'desc_length', 'country']]]
y = df['fraudulent'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Total Dataset:      {len(X):,} listings")
print(f"Training Partition:  {len(X_train):,} listings ({sum(y_train == 0):,} Legit, {sum(y_train == 1):,} Fraud)")
print(f"Testing Partition:   {len(X_test):,} listings ({sum(y_test == 0):,} Legit, {sum(y_test == 1):,} Fraud)")
print(f"Training Fraud %:    {sum(y_train == 1) / len(y_train) * 100:.2f}%")
print(f"Testing Fraud %:     {sum(y_test == 1) / len(y_test) * 100:.2f}%")


## 8. Full Scikit-Learn Production Pipeline Construction
Construct a composite `FeatureUnion` combining TF-IDF word & character n-grams with structured metadata, feeding into a class-balanced Logistic Regression classifier.


In [ ]:
# Pipeline Construction Factory Functions
def build_feature_extractor():
    text_pipeline = Pipeline([
        ("combiner", TextCombiner()),
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            min_df=3,
            sublinear_tf=True
        ))
    ])

    structured_pipeline = Pipeline([
        ("extractor", StructuredFeatureExtractor())
    ])

    return FeatureUnion([
        ("text", text_pipeline),
        ("structured", structured_pipeline)
    ])

def build_production_pipeline():
    features = build_feature_extractor()
    classifier = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        C=1.0,
        random_state=RANDOM_STATE,
        solver="liblinear"
    )
    return Pipeline([
        ("features", features),
        ("classifier", classifier)
    ])

prod_pipeline = build_production_pipeline()
print("Production Pipeline constructed successfully:")
print(prod_pipeline)


## 9. Stratified 5-Fold Cross-Validation
Evaluate pipeline generalization across 5 cross-validation folds on the training data.


In [ ]:
# Stratified 5-Fold Cross-Validation
print("Running 5-Fold Stratified Cross-Validation on training data...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(prod_pipeline, X_train, y_train, cv=cv, scoring="f1")

print(f"5-Fold CV F1-Scores: {[round(s, 4) for s in cv_scores]}")
print(f"Mean CV F1-Score:    {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")


## 10. Primary Model Training & Held-Out Test Evaluation
Fit the production pipeline on `X_train` and evaluate on `X_test`.


In [ ]:
# Fit Primary Production Pipeline
print("Fitting primary production pipeline on full X_train...")
t0 = time.time()
prod_pipeline.fit(X_train, y_train)
train_time = time.time() - t0
print(f"Training completed in {train_time:.2f} seconds.")

# Predict on held-out test partition
y_pred_lr = prod_pipeline.predict(X_test)
y_prob_lr = prod_pipeline.predict_proba(X_test)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr, pos_label=1)
rec_lr = recall_score(y_test, y_pred_lr, pos_label=1)
f1_lr = f1_score(y_test, y_pred_lr, pos_label=1)
roc_lr = roc_auc_score(y_test, y_prob_lr)
pr_lr = average_precision_score(y_test, y_prob_lr)

print("=" * 60)
print("TEST EVALUATION METRICS (PRODUCTION LOGISTIC REGRESSION)")
print("=" * 60)
print(f"Accuracy:        {acc_lr * 100:.2f}%")
print(f"Fraud Precision: {prec_lr * 100:.2f}%")
print(f"Fraud Recall:    {rec_lr * 100:.2f}%  <-- Caught {int(rec_lr * sum(y_test))} of {sum(y_test)} test scams")
print(f"Fraud F1-Score:  {f1_lr:.4f}")
print(f"ROC-AUC:         {roc_lr:.4f}")
print(f"PR-AUC:          {pr_lr:.4f}")
print("=" * 60)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=["Legitimate (0)", "Fraudulent (1)"]))


## 11. Multi-Model Benchmark Comparison
Compare 4 candidate models:
1. **Balanced Logistic Regression** (Production Pipeline)
2. **Calibrated Linear SVM**
3. **Multinomial Naive Bayes**
4. **Random Forest Classifier** (on structured features)


In [ ]:
# Benchmark Comparisons across 4 Models
comparison_records = [
    {
        "Model": "Balanced Logistic Regression (Production)",
        "Accuracy": acc_lr,
        "Fraud Precision": prec_lr,
        "Fraud Recall": rec_lr,
        "Fraud F1": f1_lr,
        "ROC-AUC": roc_lr,
        "PR-AUC": pr_lr
    }
]

# Benchmark 2: Calibrated Linear SVM
try:
    print("Training Calibrated Linear SVM benchmark...")
    svm_pipe = Pipeline([
        ("features", build_feature_extractor()),
        ("classifier", CalibratedClassifierCV(estimator=LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=2000)))
    ])
    svm_pipe.fit(X_train, y_train)
    y_pred_svm = svm_pipe.predict(X_test)
    y_prob_svm = svm_pipe.predict_proba(X_test)[:, 1]
    comparison_records.append({
        "Model": "Calibrated Linear SVM",
        "Accuracy": accuracy_score(y_test, y_pred_svm),
        "Fraud Precision": precision_score(y_test, y_pred_svm, pos_label=1),
        "Fraud Recall": recall_score(y_test, y_pred_svm, pos_label=1),
        "Fraud F1": f1_score(y_test, y_pred_svm, pos_label=1),
        "ROC-AUC": roc_auc_score(y_test, y_prob_svm),
        "PR-AUC": average_precision_score(y_test, y_prob_svm)
    })
except Exception as e:
    print(f"SVM comparison skipped: {e}")

# Benchmark 3: Multinomial Naive Bayes
try:
    print("Training Multinomial Naive Bayes benchmark...")
    nb_pipe = Pipeline([
        ("combiner", TextCombiner()),
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("classifier", MultinomialNB())
    ])
    nb_pipe.fit(X_train, y_train)
    y_pred_nb = nb_pipe.predict(X_test)
    y_prob_nb = nb_pipe.predict_proba(X_test)[:, 1]
    comparison_records.append({
        "Model": "Multinomial Naive Bayes",
        "Accuracy": accuracy_score(y_test, y_pred_nb),
        "Fraud Precision": precision_score(y_test, y_pred_nb, pos_label=1),
        "Fraud Recall": recall_score(y_test, y_pred_nb, pos_label=1),
        "Fraud F1": f1_score(y_test, y_pred_nb, pos_label=1),
        "ROC-AUC": roc_auc_score(y_test, y_prob_nb),
        "PR-AUC": average_precision_score(y_test, y_prob_nb)
    })
except Exception as e:
    print(f"Naive Bayes comparison skipped: {e}")

# Benchmark 4: Random Forest on Structured Features
try:
    print("Training Random Forest benchmark on structured features...")
    rf_pipe = Pipeline([
        ("extractor", StructuredFeatureExtractor()),
        ("classifier", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE))
    ])
    rf_pipe.fit(X_train, y_train)
    y_pred_rf = rf_pipe.predict(X_test)
    y_prob_rf = rf_pipe.predict_proba(X_test)[:, 1]
    comparison_records.append({
        "Model": "Random Forest (Structured Features)",
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Fraud Precision": precision_score(y_test, y_pred_rf, pos_label=1),
        "Fraud Recall": recall_score(y_test, y_pred_rf, pos_label=1),
        "Fraud F1": f1_score(y_test, y_pred_rf, pos_label=1),
        "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
        "PR-AUC": average_precision_score(y_test, y_prob_rf)
    })
except Exception as e:
    print(f"Random Forest comparison skipped: {e}")

comp_df = pd.DataFrame(comparison_records)
comp_df.to_csv("model_comparison.csv", index=False)
print("\nAlgorithm Benchmark Comparison Table:")
display(comp_df.round(4))

# Bar plot of comparisons
comp_df.set_index('Model')[['Fraud Precision', 'Fraud Recall', 'Fraud F1']].plot(kind='bar', figsize=(11, 5), colormap='viridis')
plt.title('Benchmark Performance Comparison on Fraudulent Class', fontweight='bold')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 12. Diagnostic Visualizations: Confusion Matrix, ROC & PR Curves
Generate diagnostic plots for your presentation slides and research report.


In [ ]:
# Diagnostic Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr', cbar=False, ax=axes[0],
            xticklabels=['Pred Legit (0)', 'Pred Fraud (1)'],
            yticklabels=['Actual Legit (0)', 'Actual Fraud (1)'])
axes[0].set_title(f'Confusion Matrix (Production)\nTotal Test: {len(y_test):,}', fontweight='bold')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# 2. ROC Curves Comparison
axes[1].plot(*roc_curve(y_test, y_prob_lr)[:2], color='#3B82F6', lw=2, label=f'Logistic Regression (AUC={roc_lr:.3f})')
if 'y_prob_svm' in locals():
    axes[1].plot(*roc_curve(y_test, y_prob_svm)[:2], color='#10B981', lw=2, label=f'Linear SVM (AUC={roc_auc_score(y_test, y_prob_svm):.3f})')
if 'y_prob_nb' in locals():
    axes[1].plot(*roc_curve(y_test, y_prob_nb)[:2], color='#F59E0B', lw=2, label=f'Naive Bayes (AUC={roc_auc_score(y_test, y_prob_nb):.3f})')
axes[1].plot([0, 1], [0, 1], color='#94A3B8', linestyle='--', label='Random')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

# 3. Precision-Recall Curve Comparison
axes[2].plot(*precision_recall_curve(y_test, y_prob_lr)[:2][::-1], color='#8B5CF6', lw=2, label=f'Logistic Regression (PR={pr_lr:.3f})')
if 'y_prob_svm' in locals():
    axes[2].plot(*precision_recall_curve(y_test, y_prob_svm)[:2][::-1], color='#10B981', lw=2, label='Linear SVM')
axes[2].set_title('Precision-Recall Curve (Imbalanced Evaluation)', fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc='lower left')

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Confusion matrix graphic saved to 'confusion_matrix.png'.")


## 13. Explainable AI (XAI) Feature Attribution
Inspect vocabulary weights from the fitted model. In linear models with TF-IDF, positive coefficients push predictions toward Fraud ($w_i > 0$), while negative coefficients indicate Legitimacy ($w_i < 0$).


In [ ]:
# Feature Attribution Analysis
features_step = prod_pipeline.named_steps['features']
text_step = dict(features_step.transformer_list)['text']
tfidf_model = text_step.named_steps['tfidf']
classifier = prod_pipeline.named_steps['classifier']

feature_names = np.array(tfidf_model.get_feature_names_out())
coefs = classifier.coef_[0][:len(feature_names)]

top_fraud_idx = np.argsort(coefs)[-15:]
top_legit_idx = np.argsort(coefs)[:15]

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.barh(feature_names[top_fraud_idx], coefs[top_fraud_idx], color='#EF4444')
plt.title('Top 15 Fraud-Indicating Terms (Positive Weights)', fontweight='bold')
plt.xlabel('Logistic Regression Weight Coefficient')

plt.subplot(1, 2, 2)
plt.barh(feature_names[top_legit_idx], coefs[top_legit_idx], color='#10B981')
plt.title('Top 15 Legitimacy-Indicating Terms (Negative Weights)', fontweight='bold')
plt.xlabel('Logistic Regression Weight Coefficient')

plt.tight_layout()
plt.show()


## 14. Internship Scam Guard & Dual-Engine Risk Scoring
Combine ML probability (60%) with domain-specific heuristic rules (40%) targeting student internship scams (registration fees, Telegram contact, guaranteed jobs without interview).


In [ ]:
# Rule-Based Scam Guard & Risk Engine
SCAM_PATTERNS = {
    "Registration / Upfront Fee": [
        r"\b(registration|processing|application|training|onboarding|equipment)\s+fee\b",
        r"\bsecurity\s+deposit\b",
        r"\bpay\s+(\$|₹|rs|inr|usd)?\s*\d+\s*(to\s+apply|to\s+register|for\s+laptop)\b"
    ],
    "Guaranteed Selection / No Interview": [
        r"\b100%\s*(guaranteed|placement|job\s+offer)\b",
        r"\bno\s+interview\s+needed\b",
        r"\bdirect\s+selection\s+without\s+interview\b"
    ],
    "Unrealistic Earnings": [
        r"\bearn\s+(\$|₹|rs)?\s*\d{3,}\s*(per\s+day|daily)\b",
        r"\btyping\s+jobs?\s*-\s*earn\b",
        r"\bcopy\s+paste\s+work\b"
    ],
    "Unprofessional Contact": [
        r"\bcontact\s+(me|us)\s+on\s+(telegram|whatsapp)\b",
        r"\btelegram\s*:\s*@\w+"
    ]
}

def analyze_scam_rules(job_dict):
    text = " ".join([str(job_dict.get(k) or "") for k in ["title", "company_profile", "description", "requirements", "benefits"]]).lower()
    warnings = []
    rule_score = 0.0

    for category, patterns in SCAM_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, text):
                warnings.append(f"Heuristic Red Flag: {category}")
                rule_score += 35.0
                break

    if not job_dict.get("company_profile") or len(str(job_dict.get("company_profile")).strip()) < 20:
        warnings.append("Company Transparency: Missing or very brief company profile")
        rule_score += 15.0

    if job_dict.get("has_company_logo", 0) in (0, "0", False):
        rule_score += 10.0

    return warnings, min(rule_score, 100.0)

def predict_job(job_details):
    """End-to-end inference function matching JobGuard AI backend specifications."""
    df_input = pd.DataFrame([{
        "title": str(job_details.get("title") or ""),
        "company_profile": str(job_details.get("company_profile") or ""),
        "description": str(job_details.get("description") or ""),
        "requirements": str(job_details.get("requirements") or ""),
        "benefits": str(job_details.get("benefits") or ""),
        "telecommuting": int(job_details.get("telecommuting") or 0),
        "has_company_logo": int(job_details.get("has_company_logo") or 0),
        "has_questions": int(job_details.get("has_questions") or 0),
        "employment_type": str(job_details.get("employment_type") or ""),
        "required_experience": str(job_details.get("required_experience") or ""),
        "required_education": str(job_details.get("required_education") or ""),
        "industry": str(job_details.get("industry") or ""),
        "function": str(job_details.get("function") or "")
    }])

    # 1. ML Pipeline Inference
    ml_pred = int(prod_pipeline.predict(df_input)[0])
    ml_prob = float(prod_pipeline.predict_proba(df_input)[0][1])

    # 2. Rule-Based Heuristic Evaluation
    warnings, rule_score = analyze_scam_rules(job_details)

    # 3. Composite Risk Calculation (60% ML + 40% Rules)
    composite_risk = round((0.60 * ml_prob * 100.0) + (0.40 * rule_score), 1)

    if composite_risk >= 60.0 or ml_prob >= 0.70 or rule_score >= 60.0:
        risk_level = "HIGH RISK"
        decision = "POTENTIALLY FRAUDULENT"
    elif composite_risk >= 30.0 or ml_prob >= 0.35:
        risk_level = "MEDIUM RISK"
        decision = "POTENTIALLY FRAUDULENT" if ml_prob >= 0.50 else "LEGITIMATE (CAUTION)"
    else:
        risk_level = "LOW RISK"
        decision = "LEGITIMATE"

    return {
        "title": job_details.get("title"),
        "company": job_details.get("company_name", "Unspecified"),
        "decision": decision,
        "risk_level": risk_level,
        "composite_risk_score": composite_risk,
        "ml_fraud_probability": round(ml_prob * 100, 2),
        "rule_warning_signs": warnings,
        "disclaimer": "Informational decision-support only. Not guaranteed legal proof of fraud."
    }


## 15. Real-Time Inference Demonstration
Test the trained pipeline on 3 real-world job scenarios.


In [ ]:
# Demonstration on 3 Test Scenarios
sample_1 = {
    "title": "Junior Python Developer",
    "company_name": "CloudTech Solutions Inc",
    "company_profile": "Enterprise cloud services provider founded in 2012.",
    "description": "Looking for entry-level developers to maintain REST APIs and microservices.",
    "requirements": "B.Tech in CS/IT. Experience with Python, SQL, and Git.",
    "benefits": "Medical insurance, paid time off, and technical mentorship.",
    "location": "Bangalore, IN",
    "employment_type": "Full-time",
    "required_experience": "Entry level",
    "required_education": "Bachelor's Degree",
    "has_company_logo": 1,
    "has_questions": 1,
    "telecommuting": 0
}

sample_2 = {
    "title": "Data Entry Intern - Work from Home ($400 daily)",
    "company_name": "",
    "company_profile": "",
    "description": "Urgent requirement! Earn $400 daily for typing jobs. 100% guaranteed job offer without interview! Registration fee of $50 required before dispatching laptop. Contact on Telegram: @scamrecruiter",
    "requirements": "No experience needed. Anyone can apply.",
    "benefits": "Daily cash bonuses.",
    "location": "Remote",
    "employment_type": "Part-time",
    "has_company_logo": 0,
    "has_questions": 0,
    "telecommuting": 1
}

sample_3 = {
    "title": "Marketing Associate",
    "company_name": "Apex Promos",
    "company_profile": "",
    "description": "Promote commercial products to college campuses and online forums.",
    "requirements": "Good verbal communication skills.",
    "benefits": "Performance commissions.",
    "location": "Remote",
    "has_company_logo": 0,
    "has_questions": 0
}

for i, sample in enumerate([sample_1, sample_2, sample_3], 1):
    res = predict_job(sample)
    print(f"\nDEMO TEST {i}: {res['title']}")
    print(f"  Decision:         {res['decision']} ({res['risk_level']})")
    print(f"  Composite Risk:   {res['composite_risk_score']} / 100")
    print(f"  ML Probability:   {res['ml_fraud_probability']}%")
    print(f"  Warning Triggers: {res['rule_warning_signs'] if res['rule_warning_signs'] else 'None'}")
    print("-" * 60)


## 16. Pipeline Serialization & Google Colab Download
Save the fitted pipeline and all reports into `jobguard_ai_artifacts.zip` and trigger an automatic browser download.


In [ ]:
# Artifacts Serialization and Packaging
artifacts_dir = "jobguard_ai_artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

# 1. Save fitted Scikit-Learn production pipeline
model_path = os.path.join(artifacts_dir, "jobguard_pipeline_v1.joblib")
joblib.dump(prod_pipeline, model_path)
joblib.dump(prod_pipeline, "jobguard_pipeline_v1.joblib")

# 2. Save training metrics JSON
metrics_dict = {
    "model_name": "TF-IDF + Structured Features Logistic Regression (Balanced)",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "dataset_version": "EMSCAD Fake Job Postings",
    "total_samples": int(n_total),
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "accuracy": round(float(acc_lr), 4),
    "precision_fraud": round(float(prec_lr), 4),
    "recall_fraud": round(float(rec_lr), 4),
    "f1_fraud": round(float(f1_lr), 4),
    "roc_auc": round(float(roc_lr), 4),
    "pr_auc": round(float(pr_lr), 4),
    "confusion_matrix": {
        "true_negative": int(cm[0, 0]),
        "false_positive": int(cm[0, 1]),
        "false_negative": int(cm[1, 0]),
        "true_positive": int(cm[1, 1])
    }
}

metrics_path = os.path.join(artifacts_dir, "training_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_dict, f, indent=2)

# 3. Save classification report text
report_path = os.path.join(artifacts_dir, "classification_report.txt")
with open(report_path, "w") as f:
    f.write(classification_report(y_test, y_pred_lr, target_names=["Legitimate (0)", "Fraudulent (1)"]))

# 4. Copy confusion matrix image and comparison CSV
if os.path.exists("confusion_matrix.png"):
    shutil.copy("confusion_matrix.png", os.path.join(artifacts_dir, "confusion_matrix.png"))
if os.path.exists("model_comparison.csv"):
    shutil.copy("model_comparison.csv", os.path.join(artifacts_dir, "model_comparison.csv"))
if os.path.exists("eda_summary.json"):
    shutil.copy("eda_summary.json", os.path.join(artifacts_dir, "eda_summary.json"))

# 5. Save model metadata
metadata = {
    "project_name": "JobGuard AI",
    "model_filename": "jobguard_pipeline_v1.joblib",
    "feature_representation": "FeatureUnion(TF-IDF N-grams 10,000 + StructuredFeatureExtractor 7)",
    "classifier_algorithm": "LogisticRegression(class_weight='balanced', solver='liblinear')",
    "dataset_rows": n_total,
    "python_version": sys.version.split()[0],
    "sklearn_version": sklearn.__version__,
    "training_date_utc": datetime.now(timezone.utc).isoformat()
}
with open(os.path.join(artifacts_dir, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# 6. Compress artifacts into zip
zip_filename = "jobguard_ai_artifacts.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(artifacts_dir):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, arcname=file)

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print("=" * 60)
print(f"ALL ARTIFACTS EXPORTED AND PACKAGED: '{zip_filename}' ({zip_size_mb:.2f} MB)")
print("=" * 60)
for fn in os.listdir(artifacts_dir):
    sz = os.path.getsize(os.path.join(artifacts_dir, fn)) / 1024
    print(f"  - {fn:<30} ({sz:.1f} KB)")
print("=" * 60)

# Colab Download Triggers
try:
    from google.colab import files
    print("\nInitiating automatic download of 'jobguard_ai_artifacts.zip'...")
    files.download('jobguard_ai_artifacts.zip')
    print("Initiating individual download of 'jobguard_pipeline_v1.joblib'...")
    files.download('jobguard_pipeline_v1.joblib')
except Exception as e:
    print(f"\nColab download skipped (local runtime): {e}")
    print("Artifact files are saved in the current directory and in 'jobguard_ai_artifacts/'.")


## 17. Academic Presentation & Local Backend Integration
### How to Integrate the Trained Artifacts into Your JobGuard AI Project:
1. **Drop the Model File:**
   - Copy `jobguard_pipeline_v1.joblib` from `jobguard_ai_artifacts.zip` into your project repository at:
     `JobGuard-AI/backend/models/jobguard_pipeline_v1.joblib`
2. **Drop the Reports & Graphics:**
   - Copy `training_metrics.json`, `confusion_matrix.png`, and `eda_summary.json` into:
     `JobGuard-AI/backend/reports/`
3. **Run the Full-Stack Application:**
   - Backend: `python -m uvicorn backend.app.main:app --port 8000 --reload`
   - Frontend: `cd frontend && npm run dev`
   - Open your browser at `http://localhost:5173`. The system will automatically detect the fitted pipeline and show **Production ML Pipeline Loaded**!
